In [ ]:
import os
from skimage import io
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
import segmentation_models_pytorch as smp

In [ ]:
DATA_DIR = './data/reservoirs_10band/'

In [ ]:
def read_pred_ims(pred_dir, pred_suffix, valid_im_names, na_value=255):
    pred_im_paths = [os.path.join(pred_dir, n.replace('.png', pred_suffix))
                     for n in valid_im_names]
    pred_masks = np.stack([io.imread(p) for p in pred_im_paths])
    pred_masks[pred_masks==255] = 0
    return pred_masks
    

def compute_stats(preds, truth, cutoff=50):
    preds_binary = torch.Tensor(preds>cutoff).long()
    tp, fp, fn, tn = smp.metrics.get_stats(preds_binary,
                                           torch.Tensor(truth).long(),
                                            mode="binary")
    iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
    f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
    prec = smp.metrics.precision(tp, fp, fn, tn, reduction="micro")
    recall = smp.metrics.recall(tp, fp, fn, tn, reduction="micro")
    return np.array([iou, f1, prec,recall])

In [ ]:
y_valid_dir = os.path.join(DATA_DIR, 'ann_dir/val')
valid_im_names = os.listdir(y_valid_dir)
valid_im_paths = [os.path.join(y_valid_dir, n) for n in valid_im_names]

In [ ]:
valid_masks = np.stack([io.imread(p) for p in valid_im_paths])
pred_masks_10m = read_pred_ims(
    pred_dir='./data/preds/sentinel_outputs/raw_10m',
    pred_suffix='_sentinel_10m_out.tif',
    valid_im_names=valid_im_names
)
pred_masks_30m = read_pred_ims(
    pred_dir='./data/preds/sentinel_outputs/raw_30m',
    pred_suffix='_sentinel_30m_out.tif',
    valid_im_names=valid_im_names
)
pred_masks_combined = read_pred_ims(
    pred_dir='./data/preds/sentinel_outputs/final_combined',
    pred_suffix='_sentinel_combined_out.tif',
    valid_im_names=valid_im_names
)

In [ ]:
compute_stats(pred_masks_10m, valid_masks, cutoff=50)

In [ ]:
diff = 100*(np.sum(pred_masks_combined, axis=(1,2) ) / np.sum(pred_masks_10m>50, axis=(1,2)))
diff[np.isnan(diff)] = 0
diff[np.isinf(diff)] = 0
print('Max diff image:', np.argmax(diff))

In [ ]:
plt.imshow(valid_masks[81])

In [ ]:

plt.imshow(pred_masks_10m[81])

In [ ]:
plt.imshow((pred_masks_10m[81] + pred_masks_combined[81]), vmax=2)